# St. Louis Direct Route Analysis: Hierarchical + Jaccard

This notebook analyzes the selected direct routes using hierarchical clustering with Jaccard distance for each origin-destination pair.

For each route pair, the notebook answers:

- how many selected direct clusters remain
- how large each selected cluster is
- which clusters are dominant, using `cluster_size > 5`
- which weight configurations appear in the top 3 dominant clusters
- whether dominant clusters are broad across weight space or focused on specific weights

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

WEIGHT_COLUMNS = [
    "distance_weight",
    "population_weight",
    "traffic_weight",
    "airspace_weight",
]

METHOD_ORDER = ["Hierarchical Jaccard"]


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "St Louis" / "route_robustness" / "output").exists():
            return candidate
    raise FileNotFoundError("Could not find St Louis/route_robustness/output from the current working directory.")


ROOT = find_repo_root()
OUTPUT_DIR = ROOT / "St Louis" / "route_robustness" / "output"
DIRECT_ROUTES_DIR = ROOT / "St Louis" / "route_robustness" / "route_clusters" / "direct_routes"
DIRECT_ROUTES_DIR

## Load Direct Route Cluster Results

The hierarchical-Jaccard direct-route file is loaded for all three origin-destination pairs. Each row is one selected weighted route run with its original Jaccard cluster assignment.


In [ ]:
CLUSTER_FILES = {
    "Hierarchical Jaccard": {
        "pattern": "*_hierarchical_jaccard.csv",
        "cluster_col": "hierarchical_jaccard_cluster",
    },
}


def load_cluster_results() -> pd.DataFrame:
    frames = []
    for method, config in CLUSTER_FILES.items():
        paths = sorted(DIRECT_ROUTES_DIR.glob(config["pattern"]))
        if not paths:
            raise FileNotFoundError(
                f"No direct-route files found for {method} in {DIRECT_ROUTES_DIR}"
            )
        for path in paths:
            frame = pd.read_csv(path)
            frame["method"] = method
            frame["cluster_id"] = frame[config["cluster_col"]]
            frames.append(frame)
    return pd.concat(frames, ignore_index=True)


all_clusters = load_cluster_results()
all_clusters["method"] = pd.Categorical(all_clusters["method"], METHOD_ORDER, ordered=True)

print(f"Loaded {len(all_clusters):,} direct-route rows for {METHOD_ORDER[0]}.")
all_clusters[["method", "route_pair", "route_pair_label", "weight_id", "cluster_id", *WEIGHT_COLUMNS]].head()


## Shared Helper Functions

These functions keep each route section consistent. Each section filters to one route pair and then displays one table or graph per cell.

In [ ]:
def get_route_frame(route_pair: str) -> pd.DataFrame:
    route_frame = all_clusters[all_clusters["route_pair"] == route_pair].copy()
    if route_frame.empty:
        raise ValueError(f"No rows found for route_pair={route_pair!r}")
    return route_frame


def get_cluster_counts(route_frame: pd.DataFrame) -> pd.DataFrame:
    return (
        route_frame.groupby(["method", "route_pair_label"], observed=True, as_index=False)
        .agg(route_count=("route_run_id", "count"), cluster_count=("cluster_id", "nunique"))
        .sort_values("method")
    )


def get_cluster_sizes(route_frame: pd.DataFrame) -> pd.DataFrame:
    sizes = (
        route_frame.groupby(["method", "cluster_id"], observed=True, as_index=False)
        .agg(cluster_size=("route_run_id", "count"))
        .sort_values(["method", "cluster_size", "cluster_id"], ascending=[True, False, True])
    )
    route_counts = (
        route_frame.groupby(["method", "route_pair"], observed=True)
        .agg(route_pair_count=("route_run_id", "count"))
        .reset_index()
    )
    sizes = sizes.merge(route_counts, on=["method", "route_pair"], how="left")
    sizes["share_of_route_pair"] = sizes["cluster_size"] / sizes["route_pair_count"]
    return sizes


def get_dominant_cluster_ids(route_frame: pd.DataFrame, min_cluster_size: int = 5, top_n: int = 3) -> pd.DataFrame:
    sizes = get_cluster_sizes(route_frame)
    return (
        sizes[sizes["cluster_size"] > min_cluster_size]
        .sort_values(["method", "cluster_size"], ascending=[True, False])
        .groupby("method", observed=True, as_index=False)
        .head(top_n)
    )


def get_dominant_rows(route_frame: pd.DataFrame, min_cluster_size: int = 5, top_n: int = 3) -> pd.DataFrame:
    keys = get_dominant_cluster_ids(route_frame, min_cluster_size=min_cluster_size, top_n=top_n)[["method", "cluster_id"]]
    return route_frame.merge(keys, on=["method", "cluster_id"], how="inner")


def summarize_weight_focus(group: pd.DataFrame) -> pd.Series:
    means = group[WEIGHT_COLUMNS].mean()
    mins = group[WEIGHT_COLUMNS].min()
    maxs = group[WEIGHT_COLUMNS].max()
    stds = group[WEIGHT_COLUMNS].std(ddof=0)
    focus_column = means.idxmax()
    avg_within_std = float(stds.mean())

    if avg_within_std < 0.10:
        spread_label = "focused"
    elif avg_within_std < 0.18:
        spread_label = "moderately spread"
    else:
        spread_label = "broadly spread"

    result = {
        "cluster_size": len(group),
        "route_distance_km_mean": group["route_distance_km"].mean(),
        "route_distance_km_min": group["route_distance_km"].min(),
        "route_distance_km_max": group["route_distance_km"].max(),
        "weight_focus": focus_column.replace("_weight", ""),
        "dominant_mean_weight": means[focus_column],
        "mean_weight_std": means.std(ddof=0),
        "avg_within_cluster_weight_std": avg_within_std,
        "weight_spread_label": spread_label,
    }
    for column in WEIGHT_COLUMNS:
        label = column.replace("_weight", "")
        result[f"{label}_mean"] = means[column]
        result[f"{label}_min"] = mins[column]
        result[f"{label}_max"] = maxs[column]
    return pd.Series(result)


def get_dominant_weight_summary(route_frame: pd.DataFrame, min_cluster_size: int = 5, top_n: int = 3) -> pd.DataFrame:
    dominant_rows = get_dominant_rows(route_frame, min_cluster_size=min_cluster_size, top_n=top_n)
    return (
        dominant_rows.groupby(["method", "cluster_id"], observed=True)
        .apply(summarize_weight_focus)
        .reset_index()
        .sort_values(["method", "cluster_size"], ascending=[True, False])
    )


def get_dominant_weight_configurations(route_frame: pd.DataFrame, min_cluster_size: int = 5, top_n: int = 3) -> pd.DataFrame:
    dominant_rows = get_dominant_rows(route_frame, min_cluster_size=min_cluster_size, top_n=top_n)
    return dominant_rows[["method", "cluster_id", "weight_id", *WEIGHT_COLUMNS, "route_distance_km", "total_weighted_score"]].sort_values(
        ["method", "cluster_id", "distance_weight", "population_weight", "traffic_weight", "airspace_weight"]
    )

In [ ]:
def plot_cluster_sizes(route_frame: pd.DataFrame, route_title: str) -> None:
    sizes = get_cluster_sizes(route_frame)
    methods = [method for method in METHOD_ORDER if method in set(sizes["method"].astype(str))]
    fig, axes = plt.subplots(len(methods), 1, figsize=(12, 3.4 * len(methods)), sharex=False)
    if len(methods) == 1:
        axes = [axes]
    for axis, method in zip(axes, methods):
        data = sizes[sizes["method"].astype(str) == method].sort_values("cluster_size", ascending=False)
        axis.bar(data["cluster_id"], data["cluster_size"], color="#4b7f8c")
        axis.set_title(f"{route_title} - {method}")
        axis.set_ylabel("Routes")
        axis.tick_params(axis="x", labelrotation=75)
        axis.axhline(5, color="#9f2d20", linestyle="--", linewidth=1, label="size = 5")
        axis.legend(loc="upper right")
    fig.tight_layout()
    plt.show()


def plot_dominant_weight_heatmap(route_frame: pd.DataFrame, route_title: str) -> None:
    summary = get_dominant_weight_summary(route_frame)
    mean_weight_columns = [column.replace("_weight", "") + "_mean" for column in WEIGHT_COLUMNS]
    summary["cluster_label"] = summary["method"].astype(str) + " | " + summary["cluster_id"].astype(str) + " (n=" + summary["cluster_size"].astype(int).astype(str) + ")"
    matrix = summary.set_index("cluster_label")[mean_weight_columns]
    fig, axis = plt.subplots(figsize=(10, max(3, 0.45 * len(matrix))))
    image = axis.imshow(matrix.values, aspect="auto", cmap="YlGnBu", vmin=0, vmax=1)
    axis.set_title(f"Dominant Cluster Mean Weights - {route_title}")
    axis.set_xticks(np.arange(len(mean_weight_columns)))
    axis.set_xticklabels([value.replace("_mean", "").title() for value in mean_weight_columns])
    axis.set_yticks(np.arange(len(matrix.index)))
    axis.set_yticklabels(matrix.index)
    for row_index in range(matrix.shape[0]):
        for col_index in range(matrix.shape[1]):
            axis.text(col_index, row_index, f"{matrix.iloc[row_index, col_index]:.2f}", ha="center", va="center", color="#111111")
    fig.colorbar(image, ax=axis, label="Mean weight")
    fig.tight_layout()
    plt.show()


def plot_dominant_weight_scatter(route_frame: pd.DataFrame, route_title: str, method: str) -> None:
    dominant_rows = get_dominant_rows(route_frame)
    method_frame = dominant_rows[dominant_rows["method"].astype(str) == method]
    if method_frame.empty:
        print(f"No dominant clusters found for {route_title} - {method}.")
        return
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, sharey=True)
    for axis, (x_col, y_col) in zip(axes, [("distance_weight", "airspace_weight"), ("population_weight", "traffic_weight")]):
        for cluster_id, cluster_frame in method_frame.groupby("cluster_id"):
            axis.scatter(cluster_frame[x_col], cluster_frame[y_col], label=cluster_id, alpha=0.75, s=42)
        axis.set_title(f"{method}: {x_col.replace('_weight', '').title()} vs {y_col.replace('_weight', '').title()}")
        axis.set_xlabel(x_col.replace("_weight", "").title())
        axis.set_ylabel(y_col.replace("_weight", "").title())
        axis.set_xlim(-0.02, 1.02)
        axis.set_ylim(-0.02, 1.02)
        axis.grid(True, linewidth=0.4, alpha=0.5)
        axis.legend(loc="best", fontsize=8)
    fig.suptitle(f"Dominant Cluster Weight Configurations - {route_title} - {method}", y=1.03)
    fig.tight_layout()
    plt.show()


def print_route_interpretation(route_frame: pd.DataFrame, route_title: str) -> None:
    summary = get_dominant_weight_summary(route_frame)
    print(route_title)
    print("=" * len(route_title))
    for _, row in summary.iterrows():
        print(
            f"{row['method']} | {row['cluster_id']}: n={int(row['cluster_size'])}, {row['weight_spread_label']}, "
            f"highest mean weight={row['weight_focus']} ({row['dominant_mean_weight']:.2f}). "
            f"Mean weights: distance={row['distance_mean']:.2f}, population={row['population_mean']:.2f}, "
            f"traffic={row['traffic_mean']:.2f}, airspace={row['airspace_mean']:.2f}."
        )

# MidAmerica to St. Louis Lambert

This section filters all tables and plots to `midamerica_to_st_louis_lambert`. The route-level cells are separated so each output is easier to inspect before moving to the next table or graph.

In [ ]:
midamerica_lambert_frame = get_route_frame("midamerica_to_st_louis_lambert")
midamerica_lambert_frame[["method", "route_pair_label", "weight_id", "cluster_id", *WEIGHT_COLUMNS]].head()

## Cluster Counts

This table shows how many clusters each method formed for this route pair.

In [ ]:
get_cluster_counts(midamerica_lambert_frame)

## Cluster Sizes

This table lists every cluster for this route pair. Larger clusters represent route families that appear under more weight configurations.

In [ ]:
get_cluster_sizes(midamerica_lambert_frame)

## Cluster Size Plot

Each subplot is one clustering method for this route pair. The dashed line marks the `cluster_size > 5` threshold used for dominant clusters.

In [ ]:
plot_cluster_sizes(midamerica_lambert_frame, "MidAmerica to St. Louis Lambert")

## Dominant Clusters

Dominant clusters are the top 3 largest clusters per method after filtering to `cluster_size > 5`.

In [ ]:
get_dominant_cluster_ids(midamerica_lambert_frame)

## Dominant Cluster Weight Summary

This table checks whether each dominant cluster is weight-focused or spread across the tested weight grid.

In [ ]:
get_dominant_weight_summary(midamerica_lambert_frame)

## Dominant Cluster Weight Configurations

This table lists the exact weight configurations that belong to the dominant clusters.

In [ ]:
get_dominant_weight_configurations(midamerica_lambert_frame)

## Dominant Cluster Mean Weight Heatmap

The heatmap makes it easier to see whether the dominant clusters lean toward distance, population, traffic, or airspace.

In [ ]:
plot_dominant_weight_heatmap(midamerica_lambert_frame, "MidAmerica to St. Louis Lambert")

## Dominant Cluster Weight Scatter - Hierarchical Jaccard

This plot shows only `Hierarchical Jaccard` for this route pair. It separates the dominant clusters across two weight-space views.

In [ ]:
plot_dominant_weight_scatter(midamerica_lambert_frame, "MidAmerica to St. Louis Lambert", "Hierarchical Jaccard")

## Short Written Interpretation

This cell prints a compact interpretation of the dominant clusters for this route pair.

In [ ]:
print_route_interpretation(midamerica_lambert_frame, "MidAmerica to St. Louis Lambert")

# MidAmerica to Downtown St. Louis Union Station

This section filters all tables and plots to `midamerica_to_st_louis_union_station`. The route-level cells are separated so each output is easier to inspect before moving to the next table or graph.

In [ ]:
midamerica_union_frame = get_route_frame("midamerica_to_st_louis_union_station")
midamerica_union_frame[["method", "route_pair_label", "weight_id", "cluster_id", *WEIGHT_COLUMNS]].head()

## Cluster Counts

This table shows how many clusters each method formed for this route pair.

In [ ]:
get_cluster_counts(midamerica_union_frame)

## Cluster Sizes

This table lists every cluster for this route pair. Larger clusters represent route families that appear under more weight configurations.

In [ ]:
get_cluster_sizes(midamerica_union_frame)

## Cluster Size Plot

Each subplot is one clustering method for this route pair. The dashed line marks the `cluster_size > 5` threshold used for dominant clusters.

In [ ]:
plot_cluster_sizes(midamerica_union_frame, "MidAmerica to Downtown St. Louis Union Station")

## Dominant Clusters

Dominant clusters are the top 3 largest clusters per method after filtering to `cluster_size > 5`.

In [ ]:
get_dominant_cluster_ids(midamerica_union_frame)

## Dominant Cluster Weight Summary

This table checks whether each dominant cluster is weight-focused or spread across the tested weight grid.

In [ ]:
get_dominant_weight_summary(midamerica_union_frame)

## Dominant Cluster Weight Configurations

This table lists the exact weight configurations that belong to the dominant clusters.

In [ ]:
get_dominant_weight_configurations(midamerica_union_frame)

## Dominant Cluster Mean Weight Heatmap

The heatmap makes it easier to see whether the dominant clusters lean toward distance, population, traffic, or airspace.

In [ ]:
plot_dominant_weight_heatmap(midamerica_union_frame, "MidAmerica to Downtown St. Louis Union Station")

## Dominant Cluster Weight Scatter - Hierarchical Jaccard

This plot shows only `Hierarchical Jaccard` for this route pair. It separates the dominant clusters across two weight-space views.

In [ ]:
plot_dominant_weight_scatter(midamerica_union_frame, "MidAmerica to Downtown St. Louis Union Station", "Hierarchical Jaccard")

## Short Written Interpretation

This cell prints a compact interpretation of the dominant clusters for this route pair.

In [ ]:
print_route_interpretation(midamerica_union_frame, "MidAmerica to Downtown St. Louis Union Station")

# Downtown Airport to St. Louis Lambert

This section filters all tables and plots to `st_louis_downtown_airport_to_st_louis_lambert`. The route-level cells are separated so each output is easier to inspect before moving to the next table or graph.

In [ ]:
downtown_lambert_frame = get_route_frame("st_louis_downtown_airport_to_st_louis_lambert")
downtown_lambert_frame[["method", "route_pair_label", "weight_id", "cluster_id", *WEIGHT_COLUMNS]].head()

## Cluster Counts

This table shows how many clusters each method formed for this route pair.

In [ ]:
get_cluster_counts(downtown_lambert_frame)

## Cluster Sizes

This table lists every cluster for this route pair. Larger clusters represent route families that appear under more weight configurations.

In [ ]:
get_cluster_sizes(downtown_lambert_frame)

## Cluster Size Plot

Each subplot is one clustering method for this route pair. The dashed line marks the `cluster_size > 5` threshold used for dominant clusters.

In [ ]:
plot_cluster_sizes(downtown_lambert_frame, "Downtown Airport to St. Louis Lambert")

## Dominant Clusters

Dominant clusters are the top 3 largest clusters per method after filtering to `cluster_size > 5`.

In [ ]:
get_dominant_cluster_ids(downtown_lambert_frame)

## Dominant Cluster Weight Summary

This table checks whether each dominant cluster is weight-focused or spread across the tested weight grid.

In [ ]:
get_dominant_weight_summary(downtown_lambert_frame)

## Dominant Cluster Weight Configurations

This table lists the exact weight configurations that belong to the dominant clusters.

In [ ]:
get_dominant_weight_configurations(downtown_lambert_frame)

## Dominant Cluster Mean Weight Heatmap

The heatmap makes it easier to see whether the dominant clusters lean toward distance, population, traffic, or airspace.

In [ ]:
plot_dominant_weight_heatmap(downtown_lambert_frame, "Downtown Airport to St. Louis Lambert")

## Dominant Cluster Weight Scatter - Hierarchical Jaccard

This plot shows only `Hierarchical Jaccard` for this route pair. It separates the dominant clusters across two weight-space views.

In [ ]:
plot_dominant_weight_scatter(downtown_lambert_frame, "Downtown Airport to St. Louis Lambert", "Hierarchical Jaccard")

## Short Written Interpretation

This cell prints a compact interpretation of the dominant clusters for this route pair.

In [ ]:
print_route_interpretation(downtown_lambert_frame, "Downtown Airport to St. Louis Lambert")